In [1]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from sklearn.metrics import accuracy_score

In [2]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
print(type(X_train))
print(X_train.shape)  # (25000,)
print(X_train[0]) # 하나의 문장(리뷰)은 시작 단어 토큰 1(문장 시작을 의미하는 토큰) 로 시작 한다
print(X_train[1]) # 하나의 문장(리뷰)은 시작 단어 토큰 1 로 시작 한다

<class 'numpy.ndarray'>
(25000,)
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
[1, 194, 1153, 194, 8255, 7

In [4]:
# 리뷰 길이를 200개로 통일
# 짧은 리뷰는 0으로 채우고, 긴 리뷰는 앞부분을 잘라 200개로 맞춤
# 1차원 배열 안에 리뷰별 배열이 들어 있던 구조가 2차원 배열로 변경됨
X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)

In [5]:
print(type(X_train))
print(X_train.shape)  # (25000, 200)
print(X_train[0])
print(X_train[1])

<class 'numpy.ndarray'>
(25000, 200)
[   5   25  100   43  838  112   50  670    2    9   35  480  284    5
  150    4  172  112  167    2  336  385   39    4  172 4536 1111   17
  546   38   13  447    4  192   50   16    6  147 2025   19   14   22
    4 1920 4613  469    4   22   71   87   12   16   43  530   38   76
   15   13 1247    4   22   17  515   17   12   16  626   18    2    5
   62  386   12    8  316    8  106    5    4 2223 5244   16  480   66
 3785   33    4  130   12   16   38  619    5   25  124   51   36  135
   48   25 1415   33    6   22   12  215   28   77   52    5   14  407
   16   82    2    8    4  107  117 5952   15  256    4    2    7 3766
    5  723   36   71   43  530  476   26  400  317   46    7    4    2
 1029   13  104   88    4  381   15  297   98   32 2071   56   26  141
    6  194 7486   18    4  226   22   21  134  476   26  480    5  144
   30 5535   18   51   36   28  224   92   25  104    4  226   65   16
   38 1334   88   12   16  283    5   16

In [6]:
# 2. RNN 모델 생성
model = Sequential([
    # 입력층: 리뷰마다 단어 번호 200개 입력
    Input(shape=(200,)),
    # 단어 번호를 32개의 실수로 이루어진 벡터로 변환
    # 신경망은 실제 단어가 무엇인지는 알지 못한다. 입력된 숫자들(X) 벡터화 후 
    # 학습 과정에서 X의 벡터 패턴을 분석하여 y의 정답을 맞히도록 가중치를 조정한다
    Embedding(10000, 32),
    # 10개 뉴런으로 단어의 순서와 문맥 학습
    SimpleRNN(10),
    # 출력층: 부정(0), 긍정(1)의 확률 2개 출력
    Dense(2, activation="softmax")
])

In [7]:
# 모델 학습 설정
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
# 학습 데이터 100개씩 묶어서 전체를 5회 반복 학습
# 25000 / 100 = 250회
# 250 * 5 = 총 1250회 배치 학습
model.fit(X_train, y_train, epochs=5, batch_size=100)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.6830 - loss: 0.5762
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8470 - loss: 0.3656
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8921 - loss: 0.2771
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9076 - loss: 0.2401
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9426 - loss: 0.1630


In [9]:
# 3. 테스트 데이터 평가
# 각 리뷰에 대해 [부정 확률, 긍정 확률] 출력
print('손실값/정확도 : ', model.evaluate(X_test, y_test)) # 반환값 : [손실값, 정확도]

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8445 - loss: 0.4274
손실값/정확도 :  [0.42742934823036194, 0.8444799780845642]


In [11]:
# 4. 새로운 문장 예측 
# imdb단어가 사용하는 숫자 매핑사전 1부터시작하는 사전데이터
word_index = imdb.get_word_index()
print(list(word_index)[:10]) # 10개의 사전만 출력해서 확인
# 가져온 데이터셋의 번호는  imdb.get_word_index() 매핑사전의 값과는 차이가 있다 +3임(0, 1, 2, 3)번을 예약된 토큰으로 사용하기 위함
# 매핑사전의 값 +3을 사용한다. (0, 1, 2, 3)번을 예약된 토큰으로 사용하기 위함

review = "This movie was wonderful"  # 긍정 예시
# review = "This movie was terrible"  # 부정 예시

# IMDb단어 매칭과 동일한 단어 번호로 변환
# 하나의 문장(리뷰)은 시작 단어 토큰 1 로 시작해야 한다. 시작 토큰 번호 1 추가
review_numbers = [1] 

for word in review.lower().split(): # 소문자로 변경 후 단어별로 잘라서
    if word in word_index:
        number = word_index[word] + 3

        # 사용 범위 안에 있는 단어만 번호 유지, 데이터셋 로드 시 imdb.load_data(num_words=10000)
        if number < 10000:
            review_numbers.append(number)
        else:
            review_numbers.append(2)  # 범위 밖의 단어
    else:
        review_numbers.append(2)      # 사전에 없는 단어

# 리뷰 길이를 200개로 통일
X_new = pad_sequences([review_numbers], maxlen=200)

# 새로운 리뷰 예측
# 결과 예: [0.2, 0.8]
# [0]은 부정 확률, [1]은 긍정 확률
prob = model.predict(X_new)
print('부정0일확률/긍정1일확률 : ', prob) # [[부정0일확률, 긍정1일확률]]

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
['fawn', 'tsukino', 'nunnery', 'sonja', 'vani', 'woods', 'spiders', 'hanging', 'woody', 'trawling']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
부정0일확률/긍정1일확률 :  [[0.02014128 0.97985876]]
